# Week 9 — Simon's Algorithm, Quantum Teleportation, DFT and QFT

This notebook follows the Week 9 teaching sequence:

1. **Simon’s algorithm** — hidden XOR periodicity and interference  
2. **Quantum teleportation** — transfer of an unknown quantum state using entanglement + two classical bits  
3. **Discrete Fourier Transform (DFT)** — the classical finite-dimensional Fourier transform  
4. **Quantum Fourier Transform (QFT)** — the Fourier transform acting on quantum amplitudes  
5. **Quokka remote simulator** — export compatible quantum circuits to OpenQASM 2 and submit them remotely

The notebook is intentionally written as a **student teaching notebook**: concept → mathematics → circuit → Qiskit → interpretation → Quokka.

### Learning objectives

By the end, students should be able to:

- explain Simon's promise as exact XOR periodicity;
- derive why Simon measurements satisfy $y\cdot s=0\pmod 2$;
- recover a hidden Simon string using linear algebra over $GF(2)$;
- derive the quantum teleportation protocol from $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$;
- explain the roles of entanglement, measurement, classical communication and Pauli corrections;
- calculate a small DFT explicitly;
- derive the QFT and its controlled-phase circuit;
- distinguish DFT output values from QFT amplitudes;
- export Qiskit circuits to **OpenQASM 2** and submit suitable circuits to the **Quokka remote simulator**.

## 0. Installation

Run this cell only if the required packages are not already installed.

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib numpy requests

## 1. Imports and common helper functions

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import requests

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, state_fidelity, partial_trace
from qiskit_aer import AerSimulator
import qiskit.qasm2

backend = AerSimulator()

def run_counts(qc, shots=2048):
    compiled = transpile(qc, backend)
    return backend.run(compiled, shots=shots).result().get_counts()

def plot_counts(counts, title="Measurement counts"):
    keys = sorted(counts)
    vals = [counts[k] for k in keys]
    plt.figure(figsize=(8, 4))
    plt.bar(keys, vals)
    plt.xlabel("bit string")
    plt.ylabel("counts")
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()

# Part I — Simon's Algorithm

## 2. The promise problem

We are given oracle access to a function

$$
f:\{0,1\}^n\rightarrow\{0,1\}^n
$$

with the promise that there exists a hidden non-zero bit string

$$
s\in\{0,1\}^n,\qquad s\neq 0^n
$$

such that

$$
f(x)=f(y)
\iff
y=x\oplus s.
$$

Equivalently,

$$
\boxed{f(x)=f(x\oplus s)}
$$

for every $x$.

This is **exact XOR periodicity**. Each function value has exactly two preimages, and those two inputs differ by the same XOR mask $s$.

### Important distinction

The period is not ordinary arithmetic addition. Simon's symmetry lives in the binary vector space

$$
(\mathbb Z_2)^n.
$$

The operation is bitwise XOR:

$$
x\oplus s.
$$

## 3. Worked example: hidden period $s=110$

For $s=110$, every input is paired with the input obtained by XORing with $110$:

$$
f(x)=f(x\oplus110).
$$

For example,

$$
001\oplus110=111,
$$

so

$$
f(001)=f(111).
$$

In [ ]:
s = "110"
for x in range(8):
    xb = format(x, "03b")
    paired = "".join(str(int(a) ^ int(b)) for a, b in zip(xb, s))
    print(xb, "<->", paired)

## 4. Quantum state evolution

Simon uses two $n$-qubit registers.

Initially,

$$
|\psi_0\rangle
=
|0^n\rangle|0^n\rangle.
$$

Apply $H^{\otimes n}$ to the first register:

$$
|\psi_1\rangle
=
\frac{1}{\sqrt{2^n}}
\sum_{x\in\{0,1\}^n}
|x\rangle|0^n\rangle.
$$

The oracle acts as

$$
U_f|x\rangle|y\rangle
=
|x\rangle|y\oplus f(x)\rangle.
$$

With the second register initially $|0^n\rangle$,

$$
|\psi_2\rangle
=
\frac{1}{\sqrt{2^n}}
\sum_x
|x\rangle|f(x)\rangle.
$$

If the output register is measured as $f(x_0)$, the first register collapses to the two inputs associated with that output:

$$
|\psi_3\rangle
=
\frac{1}{\sqrt2}
\left(
|x_0\rangle+|x_0\oplus s\rangle
\right).
$$

Explicitly measuring the second register is useful for the derivation, although it is not required to obtain the final Simon distribution.

## 5. Why the second Hadamard layer reveals equations about $s$

For an $n$-bit string $x$,

$$
H^{\otimes n}|x\rangle
=
\frac{1}{\sqrt{2^n}}
\sum_{y\in\{0,1\}^n}
(-1)^{x\cdot y}|y\rangle,
$$

where the binary inner product is

$$
x\cdot y
=
x_1y_1+x_2y_2+\cdots+x_ny_n
\pmod2.
$$

Apply this to

$$
\frac{|x_0\rangle+|x_0\oplus s\rangle}{\sqrt2}.
$$

The amplitude of $|y\rangle$ is proportional to

$$
(-1)^{x_0\cdot y}
+
(-1)^{(x_0\oplus s)\cdot y}.
$$

Using

$$
(x_0\oplus s)\cdot y
=
x_0\cdot y+s\cdot y
\pmod2,
$$

we obtain

$$
(-1)^{x_0\cdot y}
\left[
1+(-1)^{s\cdot y}
\right].
$$

There are two cases:

$$
s\cdot y=0
\quad\Rightarrow\quad
1+(-1)^0=2
\qquad\text{(constructive interference)}
$$

and

$$
s\cdot y=1
\quad\Rightarrow\quad
1+(-1)^1=0
\qquad\text{(destructive interference)}.
$$

Therefore every observed string satisfies

$$
\boxed{s\cdot y=0\pmod2}.
$$

## 6. Example of the surviving states for $s=110$

Let

$$
s=110.
$$

Then

$$
s\cdot y
=
y_1+y_2
\pmod2.
$$

The allowed outcomes are

$$
000,\quad001,\quad110,\quad111,
$$

because each satisfies

$$
110\cdot y=0\pmod2.
$$

For the ideal three-qubit example the final first-register state is

$$
|\psi_4\rangle
=
\frac12
\left(
|000\rangle+|001\rangle+|110\rangle+|111\rangle
\right),
$$

up to phase conventions determined by the particular collapsed pair.

## 7. Construct a teaching oracle

For classroom use, we construct a linear function over $GF(2)$ whose kernel is exactly the span of the secret $s$.

In [ ]:
def simon_linear_matrix(secret):
    secret = np.array(secret, dtype=int)
    n = len(secret)
    if not np.any(secret):
        raise ValueError("Simon secret must be non-zero.")

    pivot = np.where(secret == 1)[0][0]
    rows = []

    for j in range(n):
        if j == pivot:
            continue
        row = np.zeros(n, dtype=int)
        row[j] = 1
        if secret[j]:
            row[pivot] = 1
        rows.append(row)

    # Keep n output qubits by appending a zero row.
    rows.append(np.zeros(n, dtype=int))
    return np.array(rows, dtype=int)

A = simon_linear_matrix([1, 1, 0])
print(A)
print("A s mod 2 =", A @ np.array([1, 1, 0]) % 2)

## 8. Convert the $GF(2)$ map to CNOT gates

In [ ]:
def simon_oracle(secret):
    secret = np.array(secret, dtype=int)
    n = len(secret)
    A = simon_linear_matrix(secret)

    qc = QuantumCircuit(2*n, name="Simon_oracle")
    for out, row in enumerate(A):
        for inp, bit in enumerate(row):
            if bit:
                qc.cx(inp, n + out)
    return qc

oracle = simon_oracle([1, 1, 0])
print(oracle.draw())

## 9. Full Simon circuit in Qiskit

In [ ]:
def simon_circuit(secret, measure=True):
    n = len(secret)
    qc = QuantumCircuit(2*n, n if measure else 0)

    qc.h(range(n))
    qc.barrier()
    qc.compose(simon_oracle(secret), inplace=True)
    qc.barrier()
    qc.h(range(n))

    if measure:
        qc.measure(range(n), range(n))

    return qc

simon_qc = simon_circuit([1, 1, 0])
print(simon_qc.draw())

simon_counts = run_counts(simon_qc, shots=4096)
print(simon_counts)
plot_counts(simon_counts, "Simon algorithm: s = 110")

## 10. Verify $y\cdot s=0$

Qiskit prints classical bit strings with the highest-index classical bit on the left.  
The following cell reverses each printed bit string back into qubit-index order before evaluating the binary dot product.

In [ ]:
secret = np.array([1, 1, 0], dtype=int)

for bitstr, count in sorted(simon_counts.items()):
    y = np.array([int(b) for b in bitstr[::-1]])
    print(
        bitstr,
        "qubit-order y =", y,
        "y·s mod 2 =", int(y @ secret % 2),
        "count =", count
    )

## 11. Classical post-processing over $GF(2)$

Each useful measurement gives one equation

$$
y^{(i)}\cdot s=0\pmod2.
$$

Stack the measured vectors as rows of a matrix $Y$:

$$
Y=
\begin{bmatrix}
- & y^{(1)} & -\\
- & y^{(2)} & -\\
\vdots & \vdots & \vdots
\end{bmatrix}.
$$

Then recover the promised non-zero vector in the null space:

$$
\boxed{Ys=0\pmod2}.
$$

For an $n$-bit secret, approximately $n-1$ independent equations determine the one-dimensional null space.

In [ ]:
def gf2_rank(M):
    if len(M) == 0:
        return 0
    A = np.array(M, dtype=np.uint8).copy() % 2
    r = 0

    for c in range(A.shape[1]):
        pivot = next((i for i in range(r, A.shape[0]) if A[i, c]), None)
        if pivot is None:
            continue

        A[[r, pivot]] = A[[pivot, r]]

        for i in range(A.shape[0]):
            if i != r and A[i, c]:
                A[i] ^= A[r]

        r += 1

    return r

def recover_secret_bruteforce(equations):
    Y = np.array(equations, dtype=int) % 2
    n = Y.shape[1]
    candidates = []

    for x in range(1, 2**n):
        candidate = np.array([(x >> q) & 1 for q in range(n)], dtype=int)
        if np.all(Y @ candidate % 2 == 0):
            candidates.append(candidate)

    return candidates

eqs = []
for bitstr in simon_counts:
    y = np.array([int(b) for b in bitstr[::-1]])

    if np.any(y) and gf2_rank(eqs + [y]) > gf2_rank(eqs):
        eqs.append(y)

    if gf2_rank(eqs) >= 2:
        break

print("Independent equations:", eqs)
print("Secret candidates:", recover_secret_bruteforce(eqs))

## 12. Complexity and conceptual lesson

Quantumly, $O(n)$ oracle samples are sufficient with high probability to obtain enough independent equations. Polynomial-time linear algebra over $GF(2)$ then recovers $s$.

In the standard randomized classical oracle model, one must discover a collision. Birthday-paradox reasoning gives an expected query scale of

$$
\Theta\left(2^{n/2}\right).
$$

Simon therefore gives an **exponential query separation**.

The important lesson is not merely “quantum parallelism.” The useful information appears because interference removes all $y$ that violate

$$
y\cdot s=0\pmod2.
$$

# Part II — Quantum Teleportation

## 13. What is teleported?

Alice possesses an unknown qubit

$$
|\psi\rangle
=
\alpha|0\rangle+\beta|1\rangle,
$$

where

$$
|\alpha|^2+|\beta|^2=1.
$$

The goal is for Bob's qubit to finish in **exactly the same state** $|\psi\rangle$.

The physical carrier of Alice's input qubit is not transported. The protocol consumes:

- one shared entangled Bell pair;
- Alice's local quantum operations and measurements;
- two classical bits sent from Alice to Bob;
- Bob's conditional Pauli corrections.

Because classical communication is required, teleportation does **not** allow faster-than-light communication.

## 14. Step 1 — Prepare the shared Bell pair

Alice and Bob share

$$
|\Phi^+\rangle_{AB}
=
\frac{|00\rangle+|11\rangle}{\sqrt2}.
$$

The full three-qubit state is

$$
|\Psi_0\rangle
=
|\psi\rangle\otimes|\Phi^+\rangle_{AB}.
$$

Substituting $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$,

$$
|\Psi_0\rangle
=
\frac{1}{\sqrt2}
\left(
\alpha|000\rangle
+\alpha|011\rangle
+\beta|100\rangle
+\beta|111\rangle
\right).
$$

Here the qubit ordering is:

$$
(\text{input},A,B).
$$

## 15. Step 2 — Alice performs the Bell-basis transformation

Alice applies

$$
\operatorname{CNOT}_{\text{input}\rightarrow A}
$$

followed by

$$
H_{\text{input}}.
$$

After the CNOT,

$$
|\Psi_1\rangle
=
\frac{1}{\sqrt2}
\left(
\alpha|000\rangle
+\alpha|011\rangle
+\beta|110\rangle
+\beta|101\rangle
\right).
$$

After the Hadamard on the input qubit, regrouping by Alice's two qubits gives

$$
|\Psi_2\rangle
=
\frac12
\Big[
|00\rangle(\alpha|0\rangle+\beta|1\rangle)
+
|01\rangle(\alpha|1\rangle+\beta|0\rangle)
$$

$$
\qquad\qquad
+
|10\rangle(\alpha|0\rangle-\beta|1\rangle)
+
|11\rangle(\alpha|1\rangle-\beta|0\rangle)
\Big].
$$

Using Pauli operators,

$$
\boxed{
|\Psi_2\rangle
=
\frac12
\left[
|00\rangle|\psi\rangle
+
|01\rangle X|\psi\rangle
+
|10\rangle Z|\psi\rangle
+
|11\rangle XZ|\psi\rangle
\right].
}
$$

## 16. Step 3 — Measurement and correction table

Alice measures her two qubits and sends the two classical bits to Bob.

| Alice result | Bob's state before correction | Bob applies | Final state |
|---|---|---|---|
| $00$ | $|\psi\rangle$ | $I$ | $|\psi\rangle$ |
| $01$ | $X|\psi\rangle$ | $X$ | $|\psi\rangle$ |
| $10$ | $Z|\psi\rangle$ | $Z$ | $|\psi\rangle$ |
| $11$ | $XZ|\psi\rangle$ | corresponding $X/Z$ corrections | $|\psi\rangle$ |

A compact correction rule is

$$
X^{m_A}Z^{m_{\psi}}
$$

with the precise bit labels determined by the circuit's classical-bit convention.

### Key point

Alice never learns $\alpha$ or $\beta$.  
The protocol works for **every valid input qubit**.

## 17. Qiskit teleportation circuit with measurement and feed-forward

This is the conceptual protocol. It uses mid-circuit measurement and classically conditioned corrections.

In [ ]:
from qiskit import QuantumRegister, ClassicalRegister

def teleportation_dynamic(theta=np.pi/3, phi=np.pi/5):
    q = QuantumRegister(3, "q")
    c0 = ClassicalRegister(1, "m0")
    c1 = ClassicalRegister(1, "m1")
    qc = QuantumCircuit(q, c0, c1)

    # Prepare an arbitrary state on q[0]:
    # RY(theta) followed by RZ(phi)
    qc.ry(theta, q[0])
    qc.rz(phi, q[0])
    qc.barrier()

    # Bell pair q[1] -- q[2]
    qc.h(q[1])
    qc.cx(q[1], q[2])
    qc.barrier()

    # Bell-basis transformation on Alice's qubits
    qc.cx(q[0], q[1])
    qc.h(q[0])

    # Alice measures
    qc.measure(q[0], c0[0])
    qc.measure(q[1], c1[0])

    # Bob's corrections
    with qc.if_test((c1, 1)):
        qc.x(q[2])
    with qc.if_test((c0, 1)):
        qc.z(q[2])

    return qc

tele_dynamic = teleportation_dynamic()
print(tele_dynamic.draw())

## 18. Teleportation without mid-circuit classical control: deferred measurement

For statevector verification and for simpler OpenQASM-2 remote execution, we can use the **principle of deferred measurement**.

Instead of measuring Alice's qubits and classically controlling Bob's corrections immediately, we coherently apply:

- a controlled-$X$ from Alice's Bell-pair qubit to Bob;
- a controlled-$Z$ from Alice's input qubit to Bob.

This circuit is unitary and leaves Bob in the teleported state before the final measurements.

In [ ]:
def teleportation_coherent(theta=np.pi/3, phi=np.pi/5, measure_all=False):
    qc = QuantumCircuit(3, 3 if measure_all else 0)

    # Unknown/example input state on q0
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    qc.barrier()

    # Shared Bell pair q1-q2
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()

    # Alice's Bell-basis transformation
    qc.cx(0, 1)
    qc.h(0)

    # Coherent form of Bob's corrections
    qc.cx(1, 2)
    qc.cz(0, 2)

    if measure_all:
        qc.measure([0, 1, 2], [0, 1, 2])

    return qc

tele_coherent = teleportation_coherent()
print(tele_coherent.draw())

## 19. Verify Bob's teleported state using the reduced density matrix

The input state is prepared independently, then compared with Bob's reduced state after the coherent teleportation circuit.

In [ ]:
theta = np.pi/3
phi = np.pi/5

input_qc = QuantumCircuit(1)
input_qc.ry(theta, 0)
input_qc.rz(phi, 0)
psi_in = Statevector.from_instruction(input_qc)

full_state = Statevector.from_instruction(
    teleportation_coherent(theta=theta, phi=phi, measure_all=False)
)

# Qiskit subsystem index 2 is Bob's qubit.
bob_state = partial_trace(full_state, [0, 1])

print("Input statevector:")
print(psi_in.data)
print("\nBob reduced density matrix:")
print(bob_state.data)
print("\nFidelity =", state_fidelity(psi_in, bob_state))

## 20. Teleportation misconceptions

1. **No particle is instantaneously transported.**
2. **Entanglement alone cannot send a controllable message.**
3. Alice does not measure $\alpha$ and $\beta$.
4. Bob needs Alice's classical information to know which correction is required.
5. The Bell pair is consumed by the protocol.
6. The no-cloning theorem is not violated: after teleportation, Alice no longer retains an independent copy of the unknown state.

# Part III — Discrete Fourier Transform (DFT)

## 21. From sample space to frequency space

A finite signal can be represented by its samples

$$
x_0,x_1,\ldots,x_{N-1},
$$

or by complex coefficients describing its discrete frequency content.

The DFT is

$$
X_k
=
\sum_{n=0}^{N-1}
x_n e^{-i2\pi kn/N},
\qquad
k=0,1,\ldots,N-1.
$$

Define the $N$th root of unity

$$
\omega_N=e^{-i2\pi/N}.
$$

Then

$$
\boxed{
X_k
=
\sum_{n=0}^{N-1}
x_n\omega_N^{kn}
}
$$

and the inverse transform is

$$
x_n
=
\frac1N
\sum_{k=0}^{N-1}
X_k e^{+i2\pi kn/N}.
$$

## 22. DFT as a matrix

The DFT can be written

$$
\mathbf X=F_N\mathbf x,
$$

where

$$
F_N=
\begin{bmatrix}
1 & 1 & 1 & \cdots & 1\\
1 & \omega_N & \omega_N^2 & \cdots & \omega_N^{N-1}\\
1 & \omega_N^2 & \omega_N^4 & \cdots & \omega_N^{2(N-1)}\\
\vdots & \vdots & \vdots & \ddots & \vdots\\
1 & \omega_N^{N-1} & \omega_N^{2(N-1)} & \cdots & \omega_N^{(N-1)^2}
\end{bmatrix}.
$$

The conventional classical DFT matrix is not normalized.  
The QFT uses the normalized unitary version with factor $1/\sqrt N$.

## 23. Explicit $N=4$ DFT

For $N=4$,

$$
\omega_4=e^{-i2\pi/4}=-i.
$$

Therefore

$$
F_4=
\begin{bmatrix}
1&1&1&1\\
1&-i&-1&i\\
1&-1&1&-1\\
1&i&-1&-i
\end{bmatrix}.
$$

For the sample vector

$$
\mathbf x=
\begin{bmatrix}
1\\
0\\
-1\\
0
\end{bmatrix},
$$

we calculate

$$
\mathbf X=F_4\mathbf x.
$$

In [ ]:
x = np.array([1, 0, -1, 0], dtype=complex)
N = len(x)

k = np.arange(N).reshape(-1, 1)
n = np.arange(N).reshape(1, -1)

F = np.exp(-2j * np.pi * k * n / N)
X_manual = F @ x
X_numpy = np.fft.fft(x)

print("DFT matrix F4:")
print(np.round(F, 4))
print("\nManual DFT:")
print(np.round(X_manual, 4))
print("\nNumPy FFT result:")
print(np.round(X_numpy, 4))

## 24. Visualise DFT magnitude and phase

In [ ]:
freq = np.arange(N)

plt.figure(figsize=(7, 4))
plt.bar(freq, np.abs(X_manual))
plt.xlabel("frequency index k")
plt.ylabel("|X[k]|")
plt.title("DFT magnitude")
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(freq, np.angle(X_manual))
plt.xlabel("frequency index k")
plt.ylabel("phase (radians)")
plt.title("DFT phase")
plt.show()

## 25. Why learn DFT before QFT?

Both DFT and QFT use the same complex phase structure.

But their interpretation is different.

### Classical DFT

Input:

$$
(x_0,x_1,\ldots,x_{N-1})
$$

Output:

$$
(X_0,X_1,\ldots,X_{N-1})
$$

and a classical computer can explicitly store/read those output coefficients.

### QFT

Input:

$$
|\psi\rangle
=
\sum_{j=0}^{N-1}a_j|j\rangle.
$$

Output:

$$
QFT|\psi\rangle
=
\sum_{k=0}^{N-1}b_k|k\rangle.
$$

The coefficients $b_k$ are **quantum amplitudes**. Measurement does not reveal the entire vector $(b_0,\ldots,b_{N-1})$.

Therefore QFT is not simply “FFT on a quantum computer.”

# Part IV — Quantum Fourier Transform

## 26. Definition

For $N=2^n$, the QFT acts on computational basis states as

$$
\boxed{
QFT_N|j\rangle
=
\frac1{\sqrt N}
\sum_{k=0}^{N-1}
e^{2\pi i jk/N}|k\rangle
}
$$

for

$$
j=0,1,\ldots,N-1.
$$

For a general state

$$
|\psi\rangle
=
\sum_{j=0}^{N-1}a_j|j\rangle,
$$

linearity gives

$$
QFT_N|\psi\rangle
=
\sum_{k=0}^{N-1}
\left[
\frac1{\sqrt N}
\sum_{j=0}^{N-1}
a_j e^{2\pi i jk/N}
\right]
|k\rangle.
$$

## 27. QFT matrix

The matrix is

$$
QFT_N
=
\frac1{\sqrt N}
\begin{bmatrix}
1&1&1&\cdots&1\\
1&\omega&\omega^2&\cdots&\omega^{N-1}\\
1&\omega^2&\omega^4&\cdots&\omega^{2(N-1)}\\
\vdots&\vdots&\vdots&\ddots&\vdots\\
1&\omega^{N-1}&\omega^{2(N-1)}&\cdots&\omega^{(N-1)^2}
\end{bmatrix},
$$

where, under the common QFT convention,

$$
\omega=e^{2\pi i/N}.
$$

The factor

$$
\frac1{\sqrt N}
$$

makes the transform unitary:

$$
QFT_N^\dagger QFT_N=I.
$$

## 28. Binary-fraction form of the QFT

Write

$$
j=j_1j_2\cdots j_n
$$

in binary.

The QFT can be factorized into single-qubit states:

$$
QFT|j_1j_2\cdots j_n\rangle
=
\frac1{2^{n/2}}
\left(
|0\rangle+e^{2\pi i\,0.j_n}|1\rangle
\right)
$$

$$
\otimes
\left(
|0\rangle+e^{2\pi i\,0.j_{n-1}j_n}|1\rangle
\right)
\otimes\cdots\otimes
\left(
|0\rangle+e^{2\pi i\,0.j_1j_2\cdots j_n}|1\rangle
\right),
$$

up to the final reversal of qubit order.

Here

$$
0.j_1j_2\cdots j_m
=
\frac{j_1}{2}
+\frac{j_2}{2^2}
+\cdots+
\frac{j_m}{2^m}.
$$

This factorization explains why the circuit is built from Hadamard gates and controlled phase rotations.

## 29. Controlled phase rotations

Define

$$
R_m
=
\begin{bmatrix}
1&0\\
0&e^{2\pi i/2^m}
\end{bmatrix}.
$$

Examples:

$$
R_2=
\begin{bmatrix}
1&0\\
0&e^{i\pi/2}
\end{bmatrix}
=
\begin{bmatrix}
1&0\\
0&i
\end{bmatrix},
$$

and

$$
R_3=
\begin{bmatrix}
1&0\\
0&e^{i\pi/4}
\end{bmatrix}.
$$

For each target qubit, the QFT uses:

1. a Hadamard to create equal-magnitude alternatives;
2. controlled phase rotations to encode binary-fraction phase;
3. final SWAP gates to reverse qubit order.

## 30. Build the QFT manually in Qiskit

In [ ]:
def qft_manual(n, do_swaps=True):
    qc = QuantumCircuit(n, name=f"QFT_{n}")

    for target in range(n - 1, -1, -1):
        qc.h(target)

        for control in range(target - 1, -1, -1):
            angle = np.pi / (2 ** (target - control))
            qc.cp(angle, control, target)

    if do_swaps:
        for i in range(n // 2):
            qc.swap(i, n - i - 1)

    return qc

qft3 = qft_manual(3)
print(qft3.draw())

## 31. Apply QFT to a computational basis state

For example, prepare

$$
|5\rangle=|101\rangle
$$

and apply the three-qubit QFT.

In [ ]:
qc = QuantumCircuit(3)

# Qiskit little-endian convention:
# integer 5 = binary 101 -> X on q0 and q2
qc.x(0)
qc.x(2)

qc.compose(qft_manual(3), inplace=True)

state = Statevector.from_instruction(qc)

print(qc.draw())
print("QFT statevector:")
for i, amp in enumerate(state.data):
    print(f"|{i:03b}> : {amp:.4f}")

## 32. Verify the QFT formula numerically

For $N=8$ and $j=5$,

$$
QFT_8|5\rangle
=
\frac1{\sqrt8}
\sum_{k=0}^{7}
e^{2\pi i(5k)/8}|k\rangle.
$$

Every basis state has magnitude

$$
\frac1{\sqrt8},
$$

but the **phases differ**.

This is an important lesson: measurement probabilities alone do not reveal the full QFT state.

In [ ]:
N = 8
j = 5
formula = np.array(
    [np.exp(2j*np.pi*j*k/N) / np.sqrt(N) for k in range(N)]
)

print("Formula amplitudes:")
for k, amp in enumerate(formula):
    print(f"|{k:03b}> : {amp:.4f}")

print("\nCircuit and formula equal up to numerical precision:",
      np.allclose(state.data, formula))

## 33. QFT followed by inverse QFT

Because QFT is unitary,

$$
QFT^\dagger QFT = I.
$$

Therefore applying QFT and then inverse QFT must recover the original state.

In [ ]:
qft = qft_manual(3)
iqft = qft.inverse()

test = QuantumCircuit(3)
test.h(0)
test.ry(0.7, 1)
test.cx(1, 2)

before = Statevector.from_instruction(test)

roundtrip = test.copy()
roundtrip.compose(qft, inplace=True)
roundtrip.compose(iqft, inplace=True)

after = Statevector.from_instruction(roundtrip)

print("State fidelity after QFT -> inverse QFT:",
      state_fidelity(before, after))

## 34. Why QFT is useful for period finding

Suppose amplitudes are concentrated on inputs separated by a period $r$:

$$
|x_0\rangle,\;
|x_0+r\rangle,\;
|x_0+2r\rangle,\ldots
$$

After QFT, the amplitude for output $k$ contains a geometric phase sum of the form

$$
\sum_m
e^{2\pi i (x_0+mr)k/N}.
$$

Factor out the $x_0$ term:

$$
e^{2\pi i x_0k/N}
\sum_m
e^{2\pi i mrk/N}.
$$

When the phases align, amplitudes add constructively. When they do not, they partially or completely cancel.

Thus periodic structure in the input becomes concentrated structure in the Fourier basis.

This interference principle is central to quantum period-finding algorithms such as Shor's algorithm.

## 35. Complexity: what QFT speed claims do and do not mean

An exact $n$-qubit QFT can be decomposed using approximately

$$
O(n^2)
$$

one- and two-qubit gates.

Since

$$
N=2^n,
$$

this is

$$
O((\log N)^2)
$$

in terms of the Hilbert-space dimension $N$.

However, this does **not** mean that QFT gives all $N$ classical Fourier coefficients exponentially faster than FFT. A measurement produces only limited classical information.

Quantum advantage occurs when an algorithm uses the QFT so that the desired **global property**—for example a period—can be inferred from a manageable number of measurements.

# Part V — DFT versus QFT

| Feature | DFT | QFT |
|---|---|---|
| Input | classical vector | amplitudes of a quantum state |
| Output | explicit classical complex coefficients | transformed quantum amplitudes |
| Normalization | convention-dependent | unitary $1/\sqrt N$ normalization |
| Directly read all outputs? | yes | no |
| Main implementation idea | arithmetic/FFT | $H$, controlled phases, SWAPs |
| Main quantum use | conceptual analogue | phase estimation, period finding, Shor, etc. |

The mathematical Fourier structure is closely related, but the computational interpretation is fundamentally different.

# Part VI — Quokka remote simulator

The following helper follows the Quokka-access pattern supplied with the class material:

1. export a Qiskit circuit as **OpenQASM 2**;
2. remove the `include "qelib1.inc";` line because the supplied Quokka example indicates that library line is not installed/needed there;
3. send JSON containing `script` and `count` to the Quokka QASM endpoint;
4. inspect the returned JSON.

> Remote access requires network connectivity and the Quokka service to be available when the cell is executed.

In [ ]:
QUOKKA_QASM_URL = "http://quokka1.quokkacomputing.com/qsim/qasm"

def to_quokka_qasm(qc):
    # Convert to a conservative basis before OpenQASM-2 export.
    # This avoids dynamic control and decomposes higher-level operations.
    basis = ["id", "x", "y", "z", "h", "s", "sdg", "t", "tdg",
             "rx", "ry", "rz", "cx", "cz", "swap", "measure", "barrier"]

    tqc = transpile(qc, basis_gates=basis, optimization_level=0)
    qasm = qiskit.qasm2.dumps(tqc)

    # The supplied Quokka notebook removes qelib1.inc.
    qasm = qasm.replace('include "qelib1.inc";\n', "")

    return qasm

def run_on_quokka(qc, count=100, timeout=30):
    qasm = to_quokka_qasm(qc)

    payload = {
        "script": qasm,
        "count": int(count)
    }

    response = requests.post(
        QUOKKA_QASM_URL,
        json=payload,
        timeout=timeout
    )
    response.raise_for_status()

    try:
        result = response.json()
    except Exception:
        result = json.loads(response.content)

    return qasm, result

def quokka_result_bits(json_obj):
    # Matches the structure used in the supplied Quokka notebook.
    values = list(json_obj["result"].values())
    return list(np.concatenate(values).flat)

## 36. Quokka — Simon's algorithm

The Simon circuit contains only Hadamards, CNOTs and final measurements, so it is a good OpenQASM-2 remote-simulator example.

In [ ]:
simon_remote = simon_circuit([1, 1, 0], measure=True)

simon_qasm = to_quokka_qasm(simon_remote)
print(simon_qasm)

# Uncomment when remote Quokka access is available:
# simon_qasm, simon_quokka_result = run_on_quokka(simon_remote, count=1000)
# print(json.dumps(simon_quokka_result, indent=2))

### Optional: analyse Quokka Simon samples

In [ ]:
# After running the previous remote call:
#
# bits = quokka_result_bits(simon_quokka_result)
# print("First returned bits:", bits[:50])
#
# Then group the returned bits according to the exact response format
# provided by the Quokka service and verify y·s = 0 (mod 2).

## 37. Quokka — Quantum teleportation

The textbook teleportation protocol uses mid-circuit measurements and classical feed-forward.  
For broad OpenQASM-2 simulator compatibility, use the **coherent/deferred-measurement** version for the remote demonstration.

To make the teleported state easy to verify by measurement, this example teleports $|1\rangle$.  
Bob should therefore be measured as `1` ideally.

In [ ]:
def teleport_one_for_quokka():
    qc = QuantumCircuit(3, 1)

    # Input |1> on q0
    qc.x(0)

    # Bell pair q1-q2
    qc.h(1)
    qc.cx(1, 2)

    # Bell-basis transformation
    qc.cx(0, 1)
    qc.h(0)

    # Deferred/coherent corrections
    qc.cx(1, 2)
    qc.cz(0, 2)

    # Measure Bob only
    qc.measure(2, 0)
    return qc

tele_remote = teleport_one_for_quokka()
print(tele_remote.draw())

tele_qasm = to_quokka_qasm(tele_remote)
print(tele_qasm)

# Uncomment when remote Quokka access is available:
# tele_qasm, tele_quokka_result = run_on_quokka(tele_remote, count=500)
# print(json.dumps(tele_quokka_result, indent=2))

## 38. Quokka — QFT

A basis-state QFT has a uniform measurement distribution, because its information is mainly in **relative phase**.  
Therefore, for a simple remote verification we apply:

$$
QFT^\dagger QFT
$$

to a known basis state. The two transforms should cancel, so measurement should recover the original basis state.

In [ ]:
def qft_roundtrip_for_quokka(n=3, value=5):
    qc = QuantumCircuit(n, n)

    # Prepare |value>
    for q in range(n):
        if (value >> q) & 1:
            qc.x(q)

    qft = qft_manual(n)
    qc.compose(qft, inplace=True)
    qc.compose(qft.inverse(), inplace=True)

    qc.measure(range(n), range(n))
    return qc

qft_remote = qft_roundtrip_for_quokka(n=3, value=5)
print(qft_remote.draw())

qft_qasm = to_quokka_qasm(qft_remote)
print(qft_qasm)

# Uncomment when remote Quokka access is available:
# qft_qasm, qft_quokka_result = run_on_quokka(qft_remote, count=500)
# print(json.dumps(qft_quokka_result, indent=2))

## 39. Why DFT itself is not sent to Quokka

The DFT section above is a **classical numerical transform**, not a quantum circuit.  
Therefore there is no meaningful DFT circuit to export to OpenQASM and execute on a quantum simulator.

The corresponding quantum circuit is the **QFT**, which is exported and submitted to Quokka in the previous section.

# Part VII — Student exercises

### Simon

1. For $s=101$, list every $y$ satisfying $y\cdot s=0\pmod2$.
2. Change the Qiskit secret from `110` to `101`.
3. Collect independent equations and recover the secret.
4. Explain why one measurement is generally insufficient.
5. Explain why the promise $f(x)=f(x\oplus s)$ must be global and exact.

### Teleportation

6. Replace the input state with $|+\rangle$.
7. Verify Bob's reduced density matrix has fidelity 1 with the input.
8. Explain why two classical bits are necessary.
9. Explain why teleportation does not violate no-cloning.
10. Compare the dynamic and deferred-measurement circuits.

### DFT

11. Calculate the DFT of $[1,1,1,1]$ by hand.
12. Calculate the DFT of $[1,0,0,0]$.
13. Explain the meaning of magnitude and phase of a DFT coefficient.

### QFT

14. Derive $QFT_4|1\rangle$ explicitly.
15. Show that every amplitude has magnitude $1/2$.
16. Build a two-qubit QFT manually.
17. Verify numerically that $QFT^\dagger QFT=I$.
18. Explain why equal measurement probabilities do not imply equal quantum states.

### Quokka

19. Export the Simon circuit to QASM and identify the `h`, `cx` and `measure` instructions.
20. Run the Simon, teleportation and QFT round-trip examples remotely when Quokka is available.

# Part VIII — Compact formula reference

## Simon

$$
f(x)=f(x\oplus s)
$$

$$
H^{\otimes n}|x\rangle
=
\frac1{\sqrt{2^n}}
\sum_y(-1)^{x\cdot y}|y\rangle
$$

$$
\boxed{y\cdot s=0\pmod2}
$$

$$
Ys=0\pmod2
$$

---

## Teleportation

$$
|\psi\rangle=\alpha|0\rangle+\beta|1\rangle
$$

$$
|\Phi^+\rangle
=
\frac{|00\rangle+|11\rangle}{\sqrt2}
$$

$$
|\Psi\rangle
=
\frac12
\left[
|00\rangle|\psi\rangle+
|01\rangle X|\psi\rangle+
|10\rangle Z|\psi\rangle+
|11\rangle XZ|\psi\rangle
\right]
$$

---

## DFT

$$
X_k
=
\sum_{n=0}^{N-1}
x_n e^{-i2\pi kn/N}
$$

$$
x_n
=
\frac1N
\sum_{k=0}^{N-1}
X_k e^{i2\pi kn/N}
$$

---

## QFT

$$
QFT_N|j\rangle
=
\frac1{\sqrt N}
\sum_{k=0}^{N-1}
e^{2\pi i jk/N}|k\rangle
$$

$$
R_m=
\begin{bmatrix}
1&0\\
0&e^{2\pi i/2^m}
\end{bmatrix}
$$

$$
QFT^\dagger QFT=I.
$$

# End of Week 9 notebook

A useful conceptual chain for students is:

**Simon:** interference reveals hidden XOR structure  
$\longrightarrow$  
**Teleportation:** entanglement + classical information transfers a quantum state  
$\longrightarrow$  
**DFT:** classical Fourier basis change  
$\longrightarrow$  
**QFT:** phase-sensitive Fourier basis change on amplitudes.

The Quokka sections then connect the classroom circuits to OpenQASM 2 and remote simulation.